# Week 47, Day 1: Where rhythms come from: gamma from excitation and inhibition

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w047/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Frequency set by inhibition

Complete one Euler step of ping(): update membrane potentials (with noise), detect and reset spikes, and update the three synaptic currents (E to I with 2 ms decay, I to E and I to I with tau_I). Then find the peak frequency of the excitatory population rate between 10 and 200 Hz for three values of tau_I.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ping(tauI=8.0, T=1000.0, dt=0.1, seed=0, NE=400, NI=100, drive=2.2, noise=0.05):
    g = np.random.default_rng(seed); p = 0.2
    W_EI = (g.random((NI, NE)) < p) * 0.15         # E -> I
    W_IE = (g.random((NE, NI)) < p) * 1.2          # I -> E
    W_II = (g.random((NI, NI)) < p) * 0.4          # I -> I
    vE, vI = g.random(NE), g.random(NI)
    xE, xIE, xII = np.zeros(NI), np.zeros(NE), np.zeros(NI)    # synaptic currents
    steps = int(T / dt); rateE = np.zeros(steps); spikes = []
    for k in range(steps):
        ...
        rateE[k] = fE.mean()
        spikes.extend((n, k * dt) for n in np.where(fE[:50])[0])   # keep spike times of 50 E cells
    return rateE, np.array(spikes)

def peak_freq(rate, dt=0.1):
    x = rate[2000:] - rate[2000:].mean()
    f = np.fft.rfftfreq(len(x), dt * 1e-3); P = np.abs(np.fft.rfft(x)) ** 2
    m = (f > 10) & (f < 200)
    return f[m][P[m].argmax()]

freq, runs = {}, {}
for tauI in [4.0, 8.0, 16.0]:
    runs[tauI] = ping(tauI); freq[tauI] = peak_freq(runs[tauI][0])
    print(f"tau_I = {tauI:4.1f} ms: peak at {freq[tauI]:.0f} Hz, mean E rate {runs[tauI][0].mean() / 1e-4:.1f} Hz")
plt.plot(np.arange(3000) * 0.1, runs[8.0][0][:3000] / 1e-4); plt.xlabel("time (ms)"); plt.ylabel("E population rate (Hz)"); plt.show()

In [ ]:
import numpy as np
def test_gamma():
    "with tau_I = 8 ms the network oscillates in the gamma band (40 to 90 Hz)"
    assert 40 < freq[8.0] < 90
def test_order():
    "frequency falls as the inhibitory decay time grows"
    assert freq[4.0] > freq[8.0] > freq[16.0]

test_gamma()
test_order()
print("All checks passed")

### Exercise: How often does each cell join in?

From the spike times of the first 50 excitatory cells after 200 ms (tau_I = 8 ms), compute each cell's interspike-interval coefficient of variation and its firing rate divided by the oscillation frequency (the fraction of cycles it joins).

In [ ]:
spk = runs[8.0][1]; spk = spk[spk[:, 1] > 200]
cvs, rates = [], []
for n in range(50):
    t = np.sort(spk[spk[:, 0] == n, 1])
    if len(t) > 5:
        ...
participation = np.mean(rates) / freq[8.0]
print(f"mean ISI CV {np.mean(cvs):.2f}; mean rate {np.mean(rates):.1f} Hz; fraction of cycles joined {participation:.2f}")

In [ ]:
import numpy as np
def test_cv():
    "single excitatory cells fire irregularly (mean ISI CV above 0.5)"
    assert np.mean(cvs) > 0.5
def test_skip():
    "each cell fires on fewer than 60% of cycles"
    assert participation < 0.6

test_cv()
test_skip()
print("All checks passed")